In [25]:
import random
from itertools import chain
import math



def fill_rooms(size, n_rooms):
    #grid = [[]]
    #for row in range(size):
    #    grid.append([])
    #    for col in range(size):
    #        grid[row].append(random.randint(1,n_rooms))

    #disconnected_rooms = check_disconnected_rooms(grid)

    #while len(disconnected_rooms)>0:
    #    grid = [[]]
    #    for row in range(size):
    #        grid.append([])
    #        for col in range(size):
    #            grid[row].append(random.randint(1,n_rooms))
    #
    #    disconnected_rooms = check_disconnected_rooms(grid)
    #    #check 
    

    grid = []
    for row in range(size):
        grid.append([])
        for col in range(size):
            grid[row].append(0)
    for room in range(1, n_rooms+1):
        cell = select_random_cell_selecting_rooms(size, grid, cannot_have_room_selected = True)
        grid[cell[0]][cell[1]] = room
        direction = random.randint(1,4)
        if direction == 1 and cell[0] > 0: 
            if grid[cell[0]-1][cell[1]] == 0:
                grid[cell[0]-1][cell[1]] = room
            else: 
                direction = 2
        if direction == 2 and cell[0] < size - 1: 
            if grid[cell[0]+1][cell[1]] == 0:
                grid[cell[0]+1][cell[1]] = room
            else: 
                direction = 3
        if direction == 3 and cell[1] > 0: 
            if grid[cell[0]][cell[1]-1] == 0:
                grid[cell[0]][cell[1]-1] = room
            else: 
                direction = 4
        if direction == 4 and cell[1] < size - 1: 
            if grid[cell[0]][cell[1]+1] == 0:
                grid[cell[0]][cell[1]+1] = room
            else: 
                direction = 1
        if direction == 1 and cell[0] > 0: 
            if grid[cell[0]-1][cell[1]] == 0:
                grid[cell[0]-1][cell[1]] = room

    while 0 in list(set(chain(*grid))):
        cell = select_random_cell_selecting_rooms(size, grid, cannot_have_room_selected = True)
        neighbor_rooms = []
        if cell[0] > 0: 
            if grid[cell[0]-1][cell[1]] != 0:
                neighbor_rooms.append(grid[cell[0]-1][cell[1]])
        if cell[0] < size - 1: 
            if grid[cell[0]+1][cell[1]] != 0:
                neighbor_rooms.append(grid[cell[0]+1][cell[1]])
        if cell[1] > 0: 
            if grid[cell[0]][cell[1]-1] != 0:
                neighbor_rooms.append(grid[cell[0]][cell[1]-1])
        if cell[1] < size - 1: 
            if grid[cell[0]][cell[1]+1] != 0:
                neighbor_rooms.append(grid[cell[0]][cell[1]+1])
        if len(neighbor_rooms) > 0:
            grid[cell[0]][cell[1]] = random.choice(neighbor_rooms)
    return grid

def select_random_cell_selecting_rooms(size, grid, cannot_have_room_selected = False):
    if not cannot_have_room_selected:
        return [random.randint(0,size-1), random.randint(0,size-1)]
    else:
        while 0 in list(set(chain(*grid))):
            cell = [random.randint(0,size-1), random.randint(0,size-1)]
            if grid[cell[0]][cell[1]] == 0:
                return cell
        return [-1, -1]

def check_disconnected_rooms(grid):
    disconnected_list = []
    room_nums = list(set(chain(*rooms)))
    for room in room_nums:
        room_locations = find_room_locations(grid, room)
        to_check = room_locations
        blobs = [[x] for x in room_locations]
        while len(to_check) > 0:
            checking = to_check[0]
            to_check.remove(checking)
            checking_blob = []
            for blob in blobs:
                if checking in blob:
                    checking_blob = blob
            blobs.remove(checking_blob)
            if [checking[0]+1, checking[1]] in room_locations:
                joining_blob = []
                for blob in blobs:
                    if [checking[0]+1, checking[1]] in blob:
                        joining_blob = blob
                        blobs.remove(joining_blob)
                    checking_blob.extend(x for x in joining_blob if x not in checking_blob)
            if [checking[0], checking[1]+1] in room_locations:
                joining_blob = []
                for blob in blobs:
                    if [checking[0], checking[1]+1] in blob:
                        joining_blob = blob
                        blobs.remove(joining_blob)
                    checking_blob.extend(x for x in joining_blob if x not in checking_blob)
            blobs.append(checking_blob)
        if len(blobs)>1:
            disconnected_list.append(room)
    return disconnected_list


def find_room_locations(grid, room_num):
    room_locations = []
    for row in range(len(grid)):
        for col in range(len(grid[row])):
            if grid[row][col] == room_num:
                room_locations.append([row, col])
    return room_locations


def count_people_in_room(rooms, room_num, person_positions):
    n_people = 0
    for [i, j] in person_positions:
        if rooms[i][j] == room_num:
            n_people += 1
    return n_people


def place_persons(size, rooms):
    not_finished = True
    room_with_victim = 0
    room_nums = list(set(chain(*rooms)))
    num_tries = 0
    while not_finished and len(room_nums) > 1 and num_tries < 1000:
        person_positions = []
        for row in range(0, size):
            double_col = True
            while double_col:
                double_col = False
                col = random.randint(0, size-1)
                for [i, j] in person_positions:
                    if j == col: 
                        double_col = True
            person_positions.append([row, col])
        for room_num in room_nums:
            if count_people_in_room(rooms, room_num, person_positions) == 2:
                not_finished = False
                room_with_victim = room_num
        num_tries += 1
    if num_tries == 1000:
        print("Impossible room, try again.")
        exit
    not_found_victim = True
    random.shuffle(person_positions)
    for [i, j] in person_positions:
        if room_with_victim == rooms[i][j] and not_found_victim:
            not_found_victim = False
            person_positions.remove([i, j])
            person_positions.append([i, j])
    return person_positions


def place_objects(rooms, objects_can_be_occupied, objects_cannot_be_occupied, person_positions, object_rate):
    object_grid = []
    for row in range(size):
        object_grid.append([])
        for col in range(size):
            object_grid[row].append(0)

    for i in range(math.floor(object_rate * len(rooms) * len(rooms))):
        object_type = random.randint(-objects_cannot_be_occupied, objects_can_be_occupied-1)  #objects not occupied get negative index, objects that can be occupied positive
        can_be_occupied = False
        if object_type >= 0:
            can_be_occupied = True
            object_type += 1
        size_hor = 1 
        size_ver = 1
        if object_type == 1 or object_type == -1:
            if random.random() < 0.5:
                size_hor = 2
            else:
                size_ver = 2
            i += 1
        if object_type == 2:
            size_hor = random.randint(1,3)
            size_ver = random.randint(1,3)
            i += size_hor * size_ver - 1

        object_position = select_random_cell_for_object(len(rooms), rooms, person_positions, object_grid, can_be_occupied, size_hor, size_ver)
        if object_position != [-1, -1]:
            for i in range(size_hor):
                for j in range(size_ver):
                    object_grid[object_position[0]+i][object_position[1]+j] = object_type
        
    return object_grid


def select_random_cell_for_object(size, rooms, person_positions, object_grid, can_be_occupied, size_hor = 1, size_ver = 1):

    num_tries = 1
    if can_be_occupied:
        occupied_or_different_rooms = False
        cell = [random.randint(0,size-size_hor), random.randint(0,size-size_ver)]
        room = rooms[cell[0]][cell[1]]
        for i in range(size_hor):
            for j in range(size_ver):
                occupied_or_different_rooms = occupied_or_different_rooms or rooms[cell[0]+i][cell[1]+j] != room or object_grid[cell[0]+i][cell[1]+j] != 0
        while occupied_or_different_rooms:
            occupied_or_different_rooms = False
            cell = [random.randint(0,size-size_hor), random.randint(0,size-size_ver)]
            room = rooms[cell[0]][cell[1]]
            for i in range(size_hor):
                for j in range(size_ver):
                    occupied_or_different_rooms = occupied_or_different_rooms or rooms[cell[0]+i][cell[1]+j] != room or object_grid[cell[0]+i][cell[1]+j] != 0
            num_tries += 1
            if num_tries > 1000:
                return [-1, -1]
        return cell
    
    else:
        occupied_or_different_rooms = False
        cell = [random.randint(0,size-size_hor), random.randint(0,size-size_ver)]
        room = rooms[cell[0]][cell[1]]
        for i in range(size_hor):
            for j in range(size_ver):
                occupied_or_different_rooms = occupied_or_different_rooms or rooms[cell[0]+i][cell[1]+j] != room or object_grid[cell[0]+i][cell[1]+j] != 0 or [cell[0]+i, cell[1]+j] in person_positions
        while occupied_or_different_rooms:
            occupied_or_different_rooms = False
            cell = [random.randint(0,size-size_hor), random.randint(0,size-size_ver)]
            #print(rooms, size_hor, size_ver, cell)
            room = rooms[cell[0]][cell[1]]
            for i in range(size_hor):
                for j in range(size_ver):
                    occupied_or_different_rooms = occupied_or_different_rooms or rooms[cell[0]+i][cell[1]+j] != room or object_grid[cell[0]+i][cell[1]+j] != 0 or [cell[0]+i, cell[1]+j] in person_positions
            num_tries += 1
            if num_tries > 1000:
                return [-1, -1]
        return cell


def generate_hints():
    return 0

def check_number_solutions():
    return 0






random.seed(2)

size = 14

n_rooms = random.randint(size - 3, size)
objects_can_be_occupied = 4 #The first is always size 2x1, the second will be randint(1,3) x randint(1,3)
objects_cannot_be_occupied = 3 #The first is always size 2x1
object_rate = 0.33

rooms = fill_rooms(size, n_rooms) 

print(f"Number of rooms: {n_rooms}")
for row in range(size):
    print(rooms[row])

print('\nLocation persons:')
person_positions = place_persons(size, rooms)
print(person_positions)

print('\nMap of objects:')
object_positions = place_objects(rooms, objects_can_be_occupied, objects_cannot_be_occupied, person_positions, object_rate)
for row in range(size):
    print(object_positions[row])


Number of rooms: 11
[1, 1, 1, 1, 9, 9, 9, 9, 9, 3, 8, 8, 8, 8]
[1, 1, 1, 1, 1, 9, 9, 9, 9, 3, 3, 8, 8, 8]
[1, 1, 1, 1, 1, 9, 9, 9, 9, 9, 3, 3, 3, 8]
[1, 1, 1, 9, 9, 9, 9, 9, 9, 3, 3, 3, 3, 3]
[1, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3, 3, 3]
[1, 1, 9, 11, 9, 9, 9, 9, 9, 3, 5, 5, 5, 5]
[1, 11, 11, 11, 9, 10, 10, 10, 7, 3, 5, 5, 5, 6]
[4, 11, 11, 11, 10, 10, 10, 7, 7, 5, 5, 5, 5, 6]
[4, 11, 11, 10, 10, 7, 7, 7, 7, 5, 5, 6, 6, 6]
[4, 11, 11, 11, 11, 6, 6, 7, 5, 5, 5, 6, 6, 6]
[4, 4, 4, 4, 4, 6, 6, 5, 5, 6, 6, 6, 6, 6]
[4, 4, 4, 4, 4, 4, 6, 6, 6, 6, 6, 6, 6, 6]
[4, 2, 4, 4, 4, 6, 6, 6, 6, 6, 6, 6, 6, 6]
[4, 2, 2, 4, 4, 4, 6, 6, 6, 6, 6, 6, 6, 6]

Location persons:
[[9, 4], [5, 9], [10, 1], [12, 12], [8, 13], [1, 2], [7, 8], [0, 10], [6, 11], [13, 7], [3, 0], [2, 6], [11, 5], [4, 3]]

Map of objects:
[-3, 0, 2, 2, 3, -2, 0, 4, 0, 0, 0, 0, 1, 1]
[2, 2, 0, -3, -2, 0, 0, 4, -2, 0, -3, -2, 0, 3]
[2, 2, 0, -1, -1, 2, 2, 2, 3, 0, -1, 4, 0, 0]
[0, -2, 0, 1, 0, 2, 2, 2, 0, -1, -1, -1, -1, -2]
[-2, 0, 0, 1